# 00. Setup check

Run every cell. Each one prints `OK` or tells you what to fix. If anything fails, raise your hand now, before we start the analysis.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

# Make the GRASS Python packages importable from this kernel.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.jupyter as gj
import grass.script as gs
from grass.tools import Tools

version = subprocess.check_output(["grass", "--version"], text=True).splitlines()[0]
print(version)
major, minor = (int(x) for x in version.split()[1].split(".")[:2])
assert (major, minor) >= (8, 5), "r.dem needs GRASS 8.5 or newer"
print("OK: GRASS version")

In [ ]:
import numpy
import scipy

print(f"numpy {numpy.__version__}, scipy {scipy.__version__}")
print("OK: Python dependencies")

In [ ]:
repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
project = repo / "grassdata" / "humbug_creek"

if project.exists():
    session = gj.init(project)
    print(f"OK: workshop project found at {project}")
else:
    # No data yet. Use a throwaway project so the tool checks below still run.
    scratch = repo / "grassdata" / "setup_check"
    if not scratch.exists():
        gs.create_project(scratch, epsg="6339")
    session = gj.init(scratch)
    print("MISSING: workshop data. Run `bash scripts/fetch_data.sh` in a terminal.")

tools = Tools()
print(tools.g_region(flags="p", format="json")["crs"])

In [ ]:
rdem = [
    "r.dem.coregister",
    "r.dem.nk",
    "r.dem.icp",
    "r.dem.dod",
    "r.dem.bias",
    "r.dem.stats",
    "r.dem.lod",
    "r.dem.change",
    "r.dem.errprop",
    "r.dem.screen",
]
missing = [name for name in rdem if shutil.which(name) is None]
if missing:
    print("MISSING:", ", ".join(missing))
    print("Install r.dem v1.1.0 as described in SETUP.md")
else:
    print("OK: all r.dem tools installed")
assert not missing